# 01 - Generate GeoRT Teacher Pseudo Labels

This notebook orchestrates real Metric3D, Depth Anything V2, DSINE, alignment, and conflict-aware fusion. Core logic lives in `src/`.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/GeoRT')
except Exception:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
%pip install -r requirements.txt

In [ ]:
import subprocess
from pathlib import Path

repos = {
    'third_party/Metric3D': ('https://github.com/YvanYin/Metric3D.git', 'hubconf.py'),
    'third_party/Depth-Anything-V2': ('https://github.com/DepthAnything/Depth-Anything-V2.git', 'depth_anything_v2/dpt.py'),
    'third_party/DSINE': ('https://github.com/baegwangbin/DSINE.git', 'projects/dsine/test_minimal.py'),
    'third_party/DMD3C': ('https://github.com/Sharpiless/DMD3C.git', 'models/BPNet.py'),
}
for dst, (url, sentinel) in repos.items():
    dst_path = Path(dst)
    if not (dst_path / sentinel).exists():
        if dst_path.exists() and sorted(p.name for p in dst_path.iterdir()) == ['.gitkeep']:
            (dst_path / '.gitkeep').unlink()
        subprocess.run(['git', 'clone', url, str(dst_path)], check=True)
    print(dst, 'OK' if (dst_path / sentinel).exists() else 'MISSING')

In [ ]:
# Required by DMD3C / BP-Net. Run on a CUDA Colab runtime.
import subprocess, sys
from pathlib import Path
ext_dir = Path('third_party/DMD3C/exts')
if ext_dir.exists():
    subprocess.run([sys.executable, 'setup.py', 'install'], cwd=str(ext_dir), check=True)


In [ ]:
from pathlib import Path

required_dirs = [
    Path('weights/metric3d'),
    Path('weights/depth_anything_v2'),
    Path('weights/dsine'),
    Path('weights/dmd3c'),
]
for d in required_dirs:
    files = sorted([p.name for p in d.glob('*')]) if d.exists() else []
    print(d, files)
assert required_dirs[0].exists(), 'Missing weights/metric3d'
assert required_dirs[1].exists(), 'Missing weights/depth_anything_v2'
assert required_dirs[2].exists(), 'Missing weights/dsine'

In [ ]:
from src.utils import load_project_config
from src.dataset import KITTIDepthCompletionDataset
from src.prepare_depth_selection import build_depth_selection_splits

cfg, paths = load_project_config('configs/teacher.yaml')
build_depth_selection_splits(paths['data_root'], train_count=800)
split = 'val'
ds = KITTIDepthCompletionDataset(paths['data_root'], paths['split_root'], paths[f'{split}_split'], split, return_tensors=False)
print('split:', split, 'samples:', len(ds))
sample = ds.load_sample_np(0)
print(sample['sample_id'], sample['rgb'].shape, sample['sparse'].shape, sample['K'])

In [ ]:
import subprocess, sys

SPLITS = ['val']
MAX_SAMPLES = None
for split in SPLITS:
    cmd = [sys.executable, '-m', 'src.teachers.generate_teachers', '--config', 'configs/teacher.yaml', '--split', split, '--run_metric3d']
    if MAX_SAMPLES is not None:
        cmd += ['--max_samples', str(MAX_SAMPLES)]
    subprocess.run(cmd, check=True)

In [ ]:
import subprocess, sys

for split in SPLITS:
    cmd = [sys.executable, '-m', 'src.teachers.generate_teachers', '--config', 'configs/teacher.yaml', '--split', split, '--run_depth_anything']
    if MAX_SAMPLES is not None:
        cmd += ['--max_samples', str(MAX_SAMPLES)]
    subprocess.run(cmd, check=True)

In [ ]:
import subprocess, sys

for split in SPLITS:
    cmd = [sys.executable, '-m', 'src.teachers.generate_teachers', '--config', 'configs/teacher.yaml', '--split', split, '--run_dsine']
    if MAX_SAMPLES is not None:
        cmd += ['--max_samples', str(MAX_SAMPLES)]
    subprocess.run(cmd, check=True)

In [ ]:
import subprocess, sys

for split in SPLITS:
    cmd = [sys.executable, '-m', 'src.teachers.generate_teachers', '--config', 'configs/teacher.yaml', '--split', split, '--run_dmd3c']
    if MAX_SAMPLES is not None:
        cmd += ['--max_samples', str(MAX_SAMPLES)]
    subprocess.run(cmd, check=True)


In [ ]:
import subprocess, sys

for split in SPLITS:
    cmd = [sys.executable, '-m', 'src.teachers.generate_teachers', '--config', 'configs/teacher.yaml', '--split', split, '--run_fusion']
    if MAX_SAMPLES is not None:
        cmd += ['--max_samples', str(MAX_SAMPLES)]
    subprocess.run(cmd, check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from src.utils import load_npz_array

sample = ds.load_sample_np(0)
sid = sample['sample_id']
root = Path(paths['teacher_root'])
D_m3d = load_npz_array(root / 'metric3d' / split / f'{sid}.npz', 'D_m3d')
D_da = load_npz_array(root / 'depth_anything' / f'{split}_aligned' / f'{sid}.npz', 'D_da_aligned')
N = load_npz_array(root / 'dsine' / split / f'{sid}.npz', 'N_dsine')
D_teacher = load_npz_array(root / 'fused' / split / f'{sid}.npz', 'D_teacher')
C_teacher = load_npz_array(root / 'fused' / split / f'{sid}.npz', 'C_teacher')

fig, ax = plt.subplots(2, 4, figsize=(18, 8))
ax[0,0].imshow(sample['rgb']); ax[0,0].set_title('RGB')
ax[0,1].imshow(sample['sparse'], cmap='magma'); ax[0,1].set_title('Sparse')
ax[0,2].imshow(D_m3d, cmap='magma'); ax[0,2].set_title('Metric3D')
ax[0,3].imshow(D_da, cmap='magma'); ax[0,3].set_title('DA aligned')
ax[1,0].imshow(((N.transpose(1,2,0) + 1) * 0.5).clip(0,1)); ax[1,0].set_title('DSINE normal')
ax[1,1].imshow(D_teacher, cmap='magma'); ax[1,1].set_title('D_teacher')
ax[1,2].imshow(C_teacher, cmap='viridis', vmin=0, vmax=1); ax[1,2].set_title('C_teacher')
ax[1,3].axis('off')
for a in ax.ravel():
    a.axis('off')
plt.tight_layout()